# 📘 Notebook 5: Inheritance

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. Understand **what inheritance is** and why it's useful
2. Create **child classes** that inherit from **parent classes**
3. Use **`super()`** to call parent class methods
4. **Override** methods in child classes
5. Understand **multiple inheritance** and the **MRO** (Method Resolution Order)
6. Use `isinstance()` and `issubclass()` for type checking

---

## 1. What is Inheritance?

**Inheritance** allows a class to **reuse** code from another class. The new class (child/subclass) automatically gets all attributes and methods from the existing class (parent/superclass).

### Real-World Analogy 🧬

Think of biological inheritance:
- A **Dog** is a type of **Animal**
- All animals have traits (name, age, species)
- Dogs inherit all animal traits AND add their own (breed, tricks)

```
        Animal (Parent/Superclass)
        ├── name, species
        ├── eat(), sleep()
        │
        ├── Dog (Child/Subclass)
        │   ├── Inherits: name, species, eat(), sleep()
        │   └── Adds: breed, fetch(), bark()
        │
        └── Cat (Child/Subclass)
            ├── Inherits: name, species, eat(), sleep()
            └── Adds: indoor, purr(), scratch()
```

### Why Use Inheritance?

| Benefit | Description |
|---------|-------------|
| **Code Reuse** | Don't repeat yourself — shared code lives in the parent |
| **Hierarchy** | Models natural "is-a" relationships |
| **Extensibility** | Add new types without modifying existing code |
| **Polymorphism** | Different types can be used interchangeably |

---

## 2. Basic Inheritance Syntax

```python
class Parent:
    # Parent class code
    pass

class Child(Parent):     # ← Child inherits from Parent
    # Child class code
    pass
```

The child class puts the parent class name in **parentheses**.

In [ ]:
class Animal:
    def __init__(self, name: str, species: str):
        self.name = name
        self.species = species
    
    def eat(self):
        print(f"{self.name} is eating.")
    
    def sleep(self):
        print(f"{self.name} is sleeping. 💤")
    
    def __repr__(self):
        return f"{self.__class__.__name__}('{self.name}', '{self.species}')"


class Dog(Animal):  # Dog inherits from Animal
    pass  # Dog gets EVERYTHING from Animal for free!


class Cat(Animal):  # Cat inherits from Animal
    pass


# Dog and Cat have all of Animal's methods and attributes
dog = Dog("Buddy", "Canine")
cat = Cat("Whiskers", "Feline")

dog.eat()    # Inherited from Animal
dog.sleep()  # Inherited from Animal
cat.eat()    # Inherited from Animal

print(f"\n{dog}")
print(cat)

---

## 3. Adding New Attributes and Methods in Child Classes

Child classes can **add** their own attributes and methods beyond what the parent provides:

In [ ]:
class Animal:
    def __init__(self, name: str, species: str):
        self.name = name
        self.species = species
    
    def eat(self):
        print(f"{self.name} is eating.")
    
    def __repr__(self):
        return f"{self.__class__.__name__}('{self.name}')"


class Dog(Animal):
    def __init__(self, name: str, breed: str):
        # Call the parent's __init__ to set name and species
        super().__init__(name, species="Canine")
        # Add Dog-specific attribute
        self.breed = breed
        self.tricks = []  # Dogs can learn tricks
    
    # Dog-specific methods
    def bark(self):
        print(f"{self.name} says: Woof! 🐕")
    
    def learn_trick(self, trick: str):
        self.tricks.append(trick)
        print(f"{self.name} learned '{trick}'!")
    
    def show_tricks(self):
        if self.tricks:
            print(f"{self.name}'s tricks: {', '.join(self.tricks)}")
        else:
            print(f"{self.name} hasn't learned any tricks yet.")


dog = Dog("Buddy", "Golden Retriever")

# Inherited from Animal
dog.eat()
print(f"Species: {dog.species}")  # Set via super().__init__

# Dog-specific
dog.bark()
dog.learn_trick("sit")
dog.learn_trick("shake")
dog.show_tricks()
print(f"Breed: {dog.breed}")

---

## 4. Understanding `super()`

**`super()`** returns a proxy object that lets you call methods on the **parent class**. It's most commonly used in `__init__` to initialize inherited attributes.

```python
class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name, species="Canine")
        #  ↑                ↑
        #  │                └── Calls Animal.__init__(self, name, "Canine")
        #  └── Refers to the parent class (Animal)
```

### Why `super()` instead of `Animal.__init__(self, ...)`?

| Approach | Pros | Cons |
|----------|------|------|
| `super().__init__(...)` | ✅ Works with multiple inheritance, ✅ DRY | Standard approach |
| `Animal.__init__(self, ...)` | Simple to read | ❌ Breaks with multiple inheritance, ❌ Hardcodes parent name |

In [ ]:
# Demonstrating super() in action
class Item:
    pay_rate = 0.8
    all = []
    
    def __init__(self, name: str, price: float, quantity: int = 0):
        assert price >= 0, f"Price {price} is not >= 0!"
        assert quantity >= 0, f"Quantity {quantity} is not >= 0!"
        self.name = name
        self.price = price
        self.quantity = quantity
        Item.all.append(self)
    
    def calculate_total_price(self):
        return self.price * self.quantity
    
    def __repr__(self):
        return f"{self.__class__.__name__}('{self.name}', {self.price}, {self.quantity})"


class Phone(Item):
    def __init__(self, name: str, price: float, quantity: int = 0, broken_phones: int = 0):
        # Call parent's __init__ to handle name, price, quantity
        super().__init__(name, price, quantity)
        
        # Phone-specific validation and attribute
        assert broken_phones >= 0, f"Broken phones {broken_phones} is not >= 0!"
        self.broken_phones = broken_phones


phone1 = Phone("iPhone 15", 999, 5, 1)
phone2 = Phone("Galaxy S24", 899, 8, 0)

print(phone1)  # Uses Item's __repr__
print(f"Broken: {phone1.broken_phones}")
print(f"Total: ${phone1.calculate_total_price()}")  # Uses Item's method

# Phone instances appear in Item.all too!
print(f"\nAll items: {Item.all}")

---

## 5. Method Overriding

A child class can **override** (replace) a parent's method by defining a method with the **same name**:

In [ ]:
class Animal:
    def __init__(self, name: str):
        self.name = name
    
    def speak(self):
        return f"{self.name} makes a generic sound."
    
    def info(self):
        return f"I am {self.name}, a {self.__class__.__name__}."


class Dog(Animal):
    def speak(self):  # OVERRIDES Animal.speak()
        return f"{self.name} says: Woof! 🐕"


class Cat(Animal):
    def speak(self):  # OVERRIDES Animal.speak()
        return f"{self.name} says: Meow! 🐱"


class Fish(Animal):
    def speak(self):  # OVERRIDES Animal.speak()
        return f"{self.name} says: ... (fish can't speak) 🐟"


# Each subclass has its own version of speak()
animals = [
    Dog("Buddy"),
    Cat("Whiskers"),
    Fish("Nemo"),
    Animal("Mystery"),  # Uses the parent version
]

for animal in animals:
    print(animal.speak())
    print(f"  → {animal.info()}")  # info() is inherited, not overridden

### Overriding with `super()` — Extending Parent Behavior

Sometimes you want to **add to** the parent's method rather than **replace** it entirely:

In [ ]:
class Logger:
    def __init__(self, name: str):
        self.name = name
        self.logs = []
    
    def log(self, message: str):
        entry = f"[{self.name}] {message}"
        self.logs.append(entry)
        return entry


class TimestampLogger(Logger):
    """Extends Logger by adding timestamps to log entries."""
    
    def log(self, message: str):
        from datetime import datetime
        timestamp = datetime.now().strftime("%H:%M:%S")
        # Call parent's log method and EXTEND it
        entry = super().log(f"{timestamp} - {message}")
        return entry


# Basic logger
basic = Logger("App")
print(basic.log("Started"))
print(basic.log("Processing"))

# Timestamp logger — extends parent's behavior
ts = TimestampLogger("Server")
print(ts.log("Started"))
print(ts.log("Request received"))

---

## 6. `isinstance()` and `issubclass()`

These functions help you check inheritance relationships:

In [ ]:
class Animal:
    pass

class Dog(Animal):
    pass

class Cat(Animal):
    pass

dog = Dog()
cat = Cat()

# isinstance() — checks if an OBJECT is an instance of a CLASS
print("=== isinstance() ===")
print(f"isinstance(dog, Dog):    {isinstance(dog, Dog)}")      # True
print(f"isinstance(dog, Animal): {isinstance(dog, Animal)}")   # True ← Dog IS an Animal!
print(f"isinstance(dog, Cat):    {isinstance(dog, Cat)}")      # False

# issubclass() — checks if a CLASS is a subclass of another CLASS
print("\n=== issubclass() ===")
print(f"issubclass(Dog, Animal): {issubclass(Dog, Animal)}")   # True
print(f"issubclass(Dog, Cat):    {issubclass(Dog, Cat)}")      # False
print(f"issubclass(Dog, Dog):    {issubclass(Dog, Dog)}")      # True (a class is its own subclass)

---

## 7. Multi-Level Inheritance

Classes can inherit from classes that also inherit from other classes:

```
Animal → Dog → GuideDog
```

In [ ]:
class Animal:
    def __init__(self, name: str):
        self.name = name
    
    def eat(self):
        return f"{self.name} is eating."


class Dog(Animal):
    def __init__(self, name: str, breed: str):
        super().__init__(name)
        self.breed = breed
    
    def bark(self):
        return f"{self.name} says: Woof!"


class GuideDog(Dog):
    def __init__(self, name: str, breed: str, handler: str):
        super().__init__(name, breed)  # Calls Dog.__init__ → which calls Animal.__init__
        self.handler = handler
    
    def guide(self):
        return f"{self.name} is guiding {self.handler}."


rex = GuideDog("Rex", "German Shepherd", "John")

# Has methods from ALL levels of the hierarchy
print(rex.eat())     # From Animal
print(rex.bark())    # From Dog
print(rex.guide())   # From GuideDog

print(f"\nName: {rex.name}, Breed: {rex.breed}, Handler: {rex.handler}")
print(f"Is Animal? {isinstance(rex, Animal)}")
print(f"Is Dog?    {isinstance(rex, Dog)}")

---

## 8. Multiple Inheritance

Python supports inheriting from **multiple** parent classes:

```python
class Child(Parent1, Parent2):
    pass
```

### Method Resolution Order (MRO)

When multiple parents have the same method, Python uses the **MRO** to decide which one to call. It follows the **C3 linearization** algorithm, but roughly:

1. The child class itself
2. Left-to-right through parent classes
3. Then grandparent classes

In [ ]:
class Flyable:
    def fly(self):
        return f"{self.name} is flying! ✈️"
    
    def move(self):
        return f"{self.name} moves by flying."


class Swimmable:
    def swim(self):
        return f"{self.name} is swimming! 🏊"
    
    def move(self):
        return f"{self.name} moves by swimming."


class Duck(Flyable, Swimmable):
    """A duck can both fly and swim!"""
    def __init__(self, name: str):
        self.name = name


donald = Duck("Donald")
print(donald.fly())    # From Flyable
print(donald.swim())   # From Swimmable

# Which move() gets called? The one from the FIRST parent (Flyable)
print(donald.move())   # Flyable.move() — because Flyable is listed first

# Check the MRO
print(f"\nMRO: {[cls.__name__ for cls in Duck.__mro__]}")

> ⚠️ **Warning**: Multiple inheritance can get complex. Prefer **composition** (using objects as attributes) over deep multiple inheritance. We'll cover this in the OOP Principles notebook.

### The Diamond Problem

When multiple parents share a common grandparent, Python's MRO ensures each class's `__init__` is called only once:

In [ ]:
class A:
    def __init__(self):
        print("A.__init__")
        super().__init__()

class B(A):
    def __init__(self):
        print("B.__init__")
        super().__init__()

class C(A):
    def __init__(self):
        print("C.__init__")
        super().__init__()

class D(B, C):  # Diamond: D → B → A and D → C → A
    def __init__(self):
        print("D.__init__")
        super().__init__()

print("Creating D():")
d = D()
print(f"\nMRO: {[cls.__name__ for cls in D.__mro__]}")
print("\n↑ Notice: A.__init__ is called only ONCE, not twice!")

---

## 9. Practical Example: E-commerce Item Hierarchy

In [ ]:
class Item:
    """Base class for all store items."""
    pay_rate = 0.8
    all = []
    
    def __init__(self, name: str, price: float, quantity: int = 0):
        assert price >= 0
        assert quantity >= 0
        self.name = name
        self.price = price
        self.quantity = quantity
        Item.all.append(self)
    
    def calculate_total_price(self):
        return self.price * self.quantity
    
    def apply_discount(self):
        self.price = self.price * self.pay_rate
    
    def __repr__(self):
        return f"{self.__class__.__name__}('{self.name}', ${self.price}, qty={self.quantity})"


class Phone(Item):
    """A phone item with additional warranty info."""
    def __init__(self, name, price, quantity=0, broken_phones=0):
        super().__init__(name, price, quantity)
        assert broken_phones >= 0
        self.broken_phones = broken_phones
    
    @property
    def working_phones(self):
        return self.quantity - self.broken_phones


class Laptop(Item):
    """A laptop item with RAM and storage specs."""
    def __init__(self, name, price, quantity=0, ram_gb=8, storage_gb=256):
        super().__init__(name, price, quantity)
        self.ram_gb = ram_gb
        self.storage_gb = storage_gb
    
    def specs(self):
        return f"{self.name}: {self.ram_gb}GB RAM, {self.storage_gb}GB Storage"


class Accessory(Item):
    """A cheap accessory — higher default discount."""
    pay_rate = 0.7  # Accessories get 30% off (overrides Item.pay_rate)


# Create items
phone = Phone("iPhone 15", 999, 10, 2)
laptop = Laptop("MacBook Pro", 2499, 5, ram_gb=16, storage_gb=512)
cable = Accessory("USB-C Cable", 20, 100)

# Use inherited + custom methods
print(f"{phone} → Working phones: {phone.working_phones}")
print(f"{laptop.specs()}")

# Demonstrate different discount rates
print(f"\nBefore discount: Cable=${cable.price}")
cable.apply_discount()  # Uses Accessory.pay_rate (0.7)
print(f"After 30% off:   Cable=${cable.price}")

# All items tracked together
print(f"\nAll items ({len(Item.all)}):")
for item in Item.all:
    print(f"  {item}")

---

## 🏋️ Practice Exercises

### Exercise 1: Vehicle Hierarchy
Create:
- `Vehicle` (parent) with `make`, `model`, `year`, `fuel_type` and a `describe()` method
- `Car(Vehicle)` with `num_doors` and `is_sedan()` method
- `Truck(Vehicle)` with `payload_capacity` and `can_haul(weight)` method
- `ElectricCar(Car)` with `battery_kwh` and `range_miles()` method

In [ ]:
# Exercise 1: Your code here



### Exercise 2: Shape Hierarchy with Method Overriding
Create:
- `Shape` (parent) with `name` and `color`, and an `area()` method that returns 0
- `Rectangle(Shape)` that overrides `area()` to compute width × height
- `Circle(Shape)` that overrides `area()` to compute π × r²
- `Square(Rectangle)` that takes only `side` and passes it as both width and height

In [ ]:
# Exercise 2: Your code here



---

### ✅ Solutions

In [ ]:
# Solution 1: Vehicle Hierarchy
class Vehicle:
    def __init__(self, make: str, model: str, year: int, fuel_type: str = "Gasoline"):
        self.make = make
        self.model = model
        self.year = year
        self.fuel_type = fuel_type
    
    def describe(self):
        return f"{self.year} {self.make} {self.model} ({self.fuel_type})"
    
    def __repr__(self):
        return f"{self.__class__.__name__}('{self.make}', '{self.model}', {self.year})"

class Car(Vehicle):
    def __init__(self, make, model, year, num_doors=4, fuel_type="Gasoline"):
        super().__init__(make, model, year, fuel_type)
        self.num_doors = num_doors
    
    def is_sedan(self):
        return self.num_doors == 4

class Truck(Vehicle):
    def __init__(self, make, model, year, payload_capacity=1000):
        super().__init__(make, model, year, "Diesel")
        self.payload_capacity = payload_capacity
    
    def can_haul(self, weight):
        return weight <= self.payload_capacity

class ElectricCar(Car):
    def __init__(self, make, model, year, battery_kwh=75, num_doors=4):
        super().__init__(make, model, year, num_doors, fuel_type="Electric")
        self.battery_kwh = battery_kwh
    
    def range_miles(self):
        return self.battery_kwh * 3.5  # Rough estimate

# Test
car = Car("Toyota", "Camry", 2024)
truck = Truck("Ford", "F-150", 2024, payload_capacity=2000)
tesla = ElectricCar("Tesla", "Model 3", 2024, battery_kwh=82)

print(car.describe())
print(f"  Sedan? {car.is_sedan()}")
print(truck.describe())
print(f"  Can haul 1500lbs? {truck.can_haul(1500)}")
print(tesla.describe())
print(f"  Range: {tesla.range_miles():.0f} miles")

In [ ]:
# Solution 2: Shape Hierarchy
import math

class Shape:
    def __init__(self, name: str, color: str = "black"):
        self.name = name
        self.color = color
    
    def area(self):
        return 0
    
    def __repr__(self):
        return f"{self.name}(color='{self.color}', area={self.area():.2f})"

class Rectangle(Shape):
    def __init__(self, width: float, height: float, color: str = "black"):
        super().__init__("Rectangle", color)
        self.width = width
        self.height = height
    
    def area(self):
        return self.width * self.height

class Circle(Shape):
    def __init__(self, radius: float, color: str = "black"):
        super().__init__("Circle", color)
        self.radius = radius
    
    def area(self):
        return math.pi * self.radius ** 2

class Square(Rectangle):
    def __init__(self, side: float, color: str = "black"):
        super().__init__(side, side, color)
        self.name = "Square"  # Override the name

shapes = [
    Rectangle(10, 5, "blue"),
    Circle(7, "red"),
    Square(4, "green"),
]

for shape in shapes:
    print(shape)

---

## 📌 Key Takeaways

1. **Inheritance** lets child classes reuse parent class code (`class Dog(Animal)`)
2. **`super().__init__(...)`** calls the parent's constructor to initialize inherited attributes
3. Child classes can **add** new attributes/methods and **override** existing ones
4. **`isinstance()`** checks objects; **`issubclass()`** checks classes
5. Python supports **multiple inheritance** with MRO determining method priority
6. Prefer **composition over inheritance** for complex relationships

---

**⏭️ Next: [Notebook 06 — Encapsulation, Getters & Setters](./06_Encapsulation_Getters_Setters.ipynb)** — Learn how to protect and control access to object data!